In [2]:
from pathlib import Path
import pandas as pd
from form990_parser import Form990Parser

%load_ext autoreload
%autoreload 2

In [3]:
file_lookup = pd.read_csv('example_usecase_lookup.csv')
file_lookup.sample(5)

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
653,A 501c3 filed a 990EZ containing IRS990Schedul...,237042361,Freelandia Institute,501c3,xml\2022\2022_01A\202241129349200214_public.xml,2021,990EZ,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
378,A 501c3 filed a 990PF containing SubstantialCo...,237337119,KANSAS ALPHA OF PHI DELTA THETA,501c3,xml\2020\2020_02A\202003219349101455_public.xml,2019,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
571,A 501c3 filed a 990 containing AffiliatedGroup...,943191447,RVM EAGLE POINT HOUSING CORPORATION,501c3,xml\2023\2023_01A\202330199349300023_public.xml,2021,990,1.0,209.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
556,A 501c3 filed a 990 containing IRS990ScheduleB...,141338342,THE YOUNG MEN'S CHRISTIAN ASSOCIATI,501c3,xml\2022\2022_01A\202241729349300004_public.xml,2021,990,1.0,237.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
705,A 501c4 filed a 990 containing IRS990ScheduleN...,471854208,Liberty Lodge 58 Real Estate Corp,501c4,xml\2024\2024_02A\202420439349302312_public.xml,2022,990,1.0,177.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
for _ in list(range(2016, 2025)):
    print(_, ",")

2016 ,
2017 ,
2018 ,
2019 ,
2020 ,
2021 ,
2022 ,
2023 ,
2024 ,


In [5]:
schedules = [
    'IRS990ScheduleC',
    'IRS990ScheduleI'
]

df = file_lookup.loc[
    (file_lookup.loc[:, schedules[0]] == 1) | (file_lookup.loc[:, schedules[1]] == 1)
]

tax_years = [
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]

df = df.loc[df.loc[:, 'TaxYr'].isin(tax_years)]

return_types = [
    '990EZ',
    '990',
    '990PF',
    '990-T'
]

if len(return_types) < 4:
    df = df.loc[df.loc[:, 'ReturnTypeCd'].isin(return_types)]
else:
    pass

org_types = [
    '501c3',
    '501c4',
    '527'
]

if len(org_types) < 3:
    df = df.loc[df.loc[:, 'OrgType'].isin(org_types)]
else:
    pass

In [6]:
import time

In [ ]:
schedule_content_accum = {
    'IRS990ScheduleC': {
        'Section527PoliticalOrgGrp': pd.DataFrame(),
        'AvgLobbyingNontaxableAmountGrp': pd.DataFrame(),
        'AvgTotalLobbyingExpendGrp': pd.DataFrame(),
        'AvgGrassrootsNontaxableGrp': pd.DataFrame(),
        'AvgGrassrootsLobbyingExpendGrp': pd.DataFrame(),
        'SupplementalInformationDetail': pd.DataFrame(),
        'Flat Content': pd.DataFrame()
    },
    'IRS990ScheduleI': {
        'RecipientTable': pd.DataFrame(),
        'GrantsOtherAsstToIndivInUSGrp': pd.DataFrame(),
        'Flat Content': pd.DataFrame()
    }
}

In [69]:
total_parse_time = 0
flat_content = dict[tuple[int, str], str]
nested_content = dict[str, list[dict[str, str | dict]]]

def flat_content_formatting(_flat_content_key: tuple[int, str], _flat_content_val: str) -> tuple[str, str]:
    return _flat_content_key[1], _flat_content_val

def nested_content_formatting(_nested_content_dict: dict):
    return _nested_content_dict['Nested Content']

for i, file in enumerate(df.file.unique()):
    example_file_path = Path(file)
    prev_time = time.time()



    parser = Form990Parser(example_file_path)

    full_formatter = parser.read_in_formatter('IRS_Form_990_Formatting.json')
    
    for schedule in schedules:
        # No need to do parsing when our reference file has already checked if it's present
        if len(df.loc[(df.file == file) & (df.loc[:, schedule] == 1)]) == 0:
            continue

        schedule_formatter = full_formatter[schedule]['Content']

        schedule_flat_targets = parser.create_xml_target_mapping_dict(
            _target_type=flat_content,
            _format_mappings_func=flat_content_formatting,
            _dict=schedule_formatter
        )

        # Nested targets
        schedule_nested_targets = parser.create_xml_target_mapping_list(
            _target_type=nested_content,
            _format_mappings_func=nested_content_formatting,
            _dict=schedule_formatter
        )

        schedule_content, missed_content = parser.parse_schedule(
            _ein='tbd',
            _tax_year='tbd',
            _schedule_root=parser.find_element(schedule),
            _flat_targets=schedule_flat_targets,
            _nested_targets=schedule_nested_targets
        )

        for content_key, content_df in schedule_content.items():
            schedule_content_accum[schedule][content_key] = pd.concat(
                [
                    schedule_content_accum[schedule][content_key],
                    content_df
                ],
                axis=0,
                ignore_index=True
            )
        if len(missed_content) > 0:
            print(file)
            print("\t", schedule)
            print("\t", missed_content)



    final_time = time.time()
    total_parse_time += final_time - prev_time
total_parse_time / (i+1)

0.2783473063641646

In [71]:
print(schedule_content_accum.keys())
schedule_content_accum['IRS990ScheduleI'].keys()

dict_keys(['IRS990ScheduleC', 'IRS990ScheduleI'])


dict_keys(['RecipientTable', 'GrantsOtherAsstToIndivInUSGrp', 'AvgTotalLobbyingExpendGrp', 'AvgGrassrootsNontaxableGrp', 'AvgGrassrootsLobbyingExpendGrp', 'SupplementalInformationDetail', 'Flat Content'])

In [73]:
schedule_content_accum['IRS990ScheduleI']['GrantsOtherAsstToIndivInUSGrp']

,FilerEIN,TaxYear,GrantType,Recipients,CashGrant,NonCashAssistance,ValuationMethodUsedDesc,NonCashAssistanceDesc
0,tbd,tbd,ECONOMIC ASSISTANCE,13,17090,0,NaN,NaN
1,tbd,tbd,SCHOLARSHIPS,25,118614,NaN,NaN,NaN
2,tbd,tbd,SCHOLARSHIP,12,36000,NaN,NaN,NaN
3,tbd,tbd,SCHOLARSHIPS,138,294339,NaN,NaN,NaN
4,tbd,tbd,EMPLOYEE CRISIS ASSISTANCE,195,290021,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
691,tbd,tbd,FELLOWSHIPS,2156,119894213,NaN,NaN,NaN
692,tbd,tbd,EMERGING LEADERS FELLOWSHIP STIPEND,9,90000,NaN,NaN,NaN
693,tbd,tbd,CHARITABLE CONTRIBUTION,841,1983592,NaN,NaN,NaN
694,tbd,tbd,SCHOLARSHIPS,2796,1397771,0,NaN,NaN
